# ApexEye SAM Service (Colab GPU)

Runs a small HTTP segmentation API in Colab and exposes it through a Cloudflare tunnel.
The ApexEye local pipeline talks to it with `src/vision/sam_client.py`.

## How to use
1. Set the runtime to GPU: **Runtime -> Change runtime type -> T4 GPU**.
2. Set `MODEL_VARIANT` and `API_KEY` in the **Config** cell.
3. Run all cells. The last cell prints a public `https://....trycloudflare.com` URL.
4. Locally set `SAM_API_URL` and `SAM_API_KEY` to those values and start ApexEye.

## Models
- `sam2.1_hiera_tiny` / `_small` / `_base_plus` / `_large` -> **Apache-2.0**, no login needed (default: tiny, fast on T4).
- `sam3` / `sam3.1` -> requires gated access on Hugging Face (`facebook/sam3.1`) and `hf auth login`.

## Protocol
- `GET/POST /health` -> `{"status": "ok", "model": ..., "device": ...}`
- `POST /segment` -> body `{"image": <base64 PNG>, "prompts": [...]}`
  - prompt types: `{"id", "type": "box", "box": [x1,y1,x2,y2]}`,
    `{"id", "type": "point", "points": [[x,y]], "labels": [1]}`,
    `{"id", "type": "text", "text": "Formula 1 car"}` (SAM 3 only)
- returns `{"masks": {"<id>": {"shape": [h,w], "counts": [...]}}, "scores": {"<id>": 0.9}}`


## 1. Config

In [ ]:
# --- Edit these ---
MODEL_VARIANT = "sam2.1_hiera_tiny"   # or sam2.1_hiera_small / _base_plus / _large, or "sam3.1"
API_KEY = "change-me-apexeye"          # shared secret; the client sends it as X-API-Key
PORT = 8000

SAM2_CONFIGS = {
    "sam2.1_hiera_tiny": ("configs/sam2.1/sam2.1_hiera_t.yaml", "sam2.1_hiera_tiny.pt"),
    "sam2.1_hiera_small": ("configs/sam2.1/sam2.1_hiera_s.yaml", "sam2.1_hiera_small.pt"),
    "sam2.1_hiera_base_plus": ("configs/sam2.1/sam2.1_hiera_b+.yaml", "sam2.1_hiera_base_plus.pt"),
    "sam2.1_hiera_large": ("configs/sam2.1/sam2.1_hiera_l.yaml", "sam2.1_hiera_large.pt"),
}
print("Configured model:", MODEL_VARIANT)

## 2. Install dependencies
Pinned to a known-good combination for the current SAM 2 repo and Colab CUDA.

In [ ]:
import torch, subprocess, sys
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# SAM 2 from source (no PyPI package)
!git clone -q https://github.com/facebookresearch/sam2.git /content/sam2
!pip install -q -e /content/sam2

# Checkpoint downloader + HTTP service
!pip install -q fastapi "uvicorn[standard]" requests
print("Dependencies installed.")

## 3. Load the model
SAM 2.1 is loaded natively. SAM 3.1 is attempted via the Hugging Face SAM 3 repo and
requires prior gated access; if it is not available the cell raises a clear error telling
you to switch back to a SAM 2.1 variant.

In [ ]:
import os, torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = None
MODEL_KIND = None


def load_sam2(variant):
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    from sam2.utils.misc import get_checkpoint_path

    config, ckpt_name = SAM2_CONFIGS[variant]
    ckpt_path = get_checkpoint_path("https://dl.fbaipublicfiles.com/segment_anything_2/092824", ckpt_name)
    model = build_sam2(config, ckpt_path, device=DEVICE)
    predictor = SAM2ImagePredictor(model)
    return predictor, "sam2"


def load_sam3(variant):
    # Gated: run `hf auth login` first and request access at huggingface.co/facebook/sam3.1
    try:
        from sam3 import build_sam3_image_model  # type: ignore
        from sam3.processor import Sam3Processor  # type: ignore
    except Exception as exc:
        raise RuntimeError(
            "SAM 3.1 is not installed or access is not granted. "
            "Request access at huggingface.co/facebook/sam3.1, run `hf auth login`, "
            "then install the sam3 package. Falling back: set MODEL_VARIANT to a sam2.1 variant."
        ) from exc
    model = build_sam3_image_model(device=DEVICE)
    return Sam3Processor(model), "sam3"


if MODEL_VARIANT in SAM2_CONFIGS:
    MODEL, MODEL_KIND = load_sam2(MODEL_VARIANT)
elif MODEL_VARIANT.startswith("sam3"):
    MODEL, MODEL_KIND = load_sam3(MODEL_VARIANT)
else:
    raise ValueError(f"Unknown MODEL_VARIANT: {MODEL_VARIANT}")

print(f"Loaded {MODEL_VARIANT} ({MODEL_KIND}) on {DEVICE}")

## 4. Segmentation helpers
RLE matches `src/vision/sam_client.py` exactly (row-major, leading zero run when the
mask starts with `True`).

In [ ]:
import base64, io
import numpy as np
from PIL import Image


def rle_encode(mask):
    flat = np.asarray(mask).astype(bool).reshape(-1)
    if flat.size == 0:
        return {"shape": [0, 0], "counts": []}
    changes = np.flatnonzero(flat[1:] != flat[:-1]) + 1
    boundaries = np.concatenate(([0], changes, [flat.size]))
    counts = np.diff(boundaries).tolist()
    if flat[0]:
        counts = [0] + counts
    return {"shape": list(np.asarray(mask).shape), "counts": counts}


def decode_image(b64):
    raw = base64.b64decode(b64)
    return np.array(Image.open(io.BytesIO(raw)).convert("RGB"))


def best_mask(masks, scores):
    masks = np.asarray(masks)
    scores = np.asarray(scores).reshape(-1)
    return masks[int(np.argmax(scores))]


def segment_sam2(predictor, image, prompts):
    predictor.set_image(image)
    out_masks, out_scores = {}, {}
    for prompt in prompts:
        kind = prompt.get("type")
        if kind == "box":
            box = np.asarray(prompt["box"], dtype=np.float32)
            masks, scores, _ = predictor.predict(box=box, multimask_output=True)
        elif kind == "point":
            pts = np.asarray(prompt["points"], dtype=np.float32)
            labels = np.asarray(prompt.get("labels", [1] * len(pts)), dtype=np.int32)
            masks, scores, _ = predictor.predict(point_coords=pts, point_labels=labels, multimask_output=True)
        else:
            raise ValueError(f"SAM 2 does not support prompt type '{kind}' (text needs SAM 3).")
        out_masks[prompt["id"]] = rle_encode(best_mask(masks, scores))
        out_scores[prompt["id"]] = float(np.max(scores))
    return out_masks, out_scores


def segment_sam3(processor, image, prompts):
    # Text prompts are the SAM 3 headline feature. Box/point prompts use the
    # processor's visual-prompt path; consult the sam3 repo for the exact
    # signature if it changes.
    state = processor.set_image(image)
    out_masks, out_scores = {}, {}
    for prompt in prompts:
        kind = prompt.get("type")
        if kind == "text":
            result = processor.set_text_prompt(prompt["text"], state)
        elif kind == "box":
            result = processor.set_box_prompt(prompt["box"], state)
        elif kind == "point":
            result = processor.set_point_prompt(prompt["points"], prompt.get("labels"), state)
        else:
            raise ValueError(f"Unsupported prompt type: {kind}")
        mask = np.asarray(result["masks"]).reshape(result["masks"].shape[-2:])
        out_masks[prompt["id"]] = rle_encode(mask)
        out_scores[prompt["id"]] = float(np.ravel(result.get("scores", [1.0]))[0])
    return out_masks, out_scores


def run_segmentation(image, prompts):
    with torch.inference_mode():
        if MODEL_KIND == "sam2":
            return segment_sam2(MODEL, image, prompts)
        return segment_sam3(MODEL, image, prompts)

print("Segmentation helpers ready.")

## 5. HTTP service (FastAPI)

In [ ]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from typing import Any, Dict, List

app = FastAPI(title="ApexEye SAM Service")


class SegmentBody(BaseModel):
    image: str
    prompts: List[Dict[str, Any]]


def check_key(api_key: str | None):
    if API_KEY and api_key != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")


@app.post("/health")
def health(api_key: str | None = Header(default=None, alias="X-API-Key")):
    check_key(api_key)
    return {"status": "ok", "model": MODEL_VARIANT, "kind": MODEL_KIND, "device": DEVICE}


@app.post("/segment")
def segment(body: SegmentBody, api_key: str | None = Header(default=None, alias="X-API-Key")):
    check_key(api_key)
    try:
        image = decode_image(body.image)
        masks, scores = run_segmentation(image, body.prompts)
        return {"masks": masks, "scores": scores}
    except Exception as exc:
        raise HTTPException(status_code=400, detail=str(exc))


print("FastAPI app defined.")

## 6. Start the server + tunnel
Starts uvicorn in a background thread, then opens a Cloudflare quick tunnel.
The printed URL changes each run; paste it into `SAM_API_URL` locally.

In [ ]:
import subprocess, threading, time, re, uvicorn


def serve():
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")


threading.Thread(target=serve, daemon=True).start()
time.sleep(3)

# Cloudflared quick tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

public_url = None
for _ in range(60):
    line = proc.stdout.readline()
    if not line:
        break
    match = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError("Could not obtain a Cloudflare tunnel URL. Re-run this cell.")

print("=" * 60)
print("SAM service is live")
print("SAM_API_URL =", public_url)
print("SAM_API_KEY =", API_KEY)
print("=" * 60)
print("Keep this Colab tab open. The tunnel closes when the session ends.")

## 7. Self-test (optional)
Verifies the service answers `/health` through the public URL.

In [ ]:
import requests
r = requests.post(f"{public_url}/health", json={}, headers={"X-API-Key": API_KEY}, timeout=30)
print(r.status_code, r.json())